In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

def process_counterfactual_csv(input_csv, output_dir="data/paired", test_size=0.15, dev_size=0.15, random_state=42):
    """
    Convert a counterfactual dataset into sentiment-labeled rows
    and split into train/dev/test TSVs under data/paired/.

    Args:
        input_csv (str): Path to input CSV
        output_dir (str): Directory for output TSV files
        test_size (float): Fraction for test split
        dev_size (float): Fraction for dev split
        random_state (int): Random seed
    """
    # Load CSV
    df = pd.read_csv(input_csv)

    # Convert into "flattened" rows
    records = []
    for idx, row in df.iterrows():
        # Original text
        records.append({
            "Sentiment": "Negative" if row["original_score"] < 0.5 else "Positive",
            "Text": row["original_text"],
            "batch_id": idx
        })
        # Counterfactual text
        records.append({
            "Sentiment": "Negative" if row["counterfactual_score"] < 0.5 else "Positive",
            "Text": row["counterfactual_text"],
            "batch_id": idx
        })

    flat_df = pd.DataFrame(records)

    # First split into train+dev and test
    train_dev, test = train_test_split(flat_df, test_size=test_size, random_state=random_state)

    # Split train_dev into train and dev
    dev_fraction = dev_size / (1 - test_size)
    train, dev = train_test_split(train_dev, test_size=dev_fraction, random_state=random_state)

    # Ensure output dir exists
    os.makedirs(output_dir, exist_ok=True)

    # Save as TSV
    train.to_csv(os.path.join(output_dir, "train_paired.tsv"), sep="\t", index=False)
    dev.to_csv(os.path.join(output_dir, "dev_paired.tsv"), sep="\t", index=False)
    test.to_csv(os.path.join(output_dir, "test_paired.tsv"), sep="\t", index=False)

    print(f"✅ Saved {len(train)} train, {len(dev)} dev, {len(test)} test examples to {output_dir}/")


# Example usage:
# process_counterfactual_csv("counterfactuals.csv")


In [3]:
import pandas as pd
import os

def process_counterfactual_csv(
    input_csv, output_dir="data/paired", test_size=0.15, dev_size=0.15
):
    """
    Convert a counterfactual dataset into sentiment-labeled rows,
    keeping original & counterfactual contiguous,
    and split sequentially (no shuffle) into train/dev/test TSVs.
    """

    # Load CSV
    df = pd.read_csv(input_csv)

    n = len(df)
    n_test = int(n * test_size)
    n_dev = int(n * dev_size)
    n_train = n - n_test - n_dev

    splits = {
        "train": df.iloc[:n_train],
        "dev": df.iloc[n_train:n_train+n_dev],
        "test": df.iloc[n_train+n_dev:],
    }

    # Ensure output dir exists
    os.makedirs(output_dir, exist_ok=True)

    for split_name, split_df in splits.items():
        records = []
        for idx, row in split_df.iterrows():
            # Original
            records.append({
                "Sentiment": "Negative" if row["original_score"] < 0.5 else "Positive",
                "Text": row["original_text"],
                "batch_id": idx
            })
            # Counterfactual
            records.append({
                "Sentiment": "Negative" if row["counterfactual_score"] < 0.5 else "Positive",
                "Text": row["counterfactual_text"],
                "batch_id": idx
            })

        flat_df = pd.DataFrame(records)

        # Save TSV with contiguous rows
        out_path = os.path.join(output_dir, f"{split_name}_paired.tsv")
        flat_df.to_csv(out_path, sep="\t", index=False)

    print(f"✅ Saved splits to {output_dir}/ (train/dev/test)")


In [4]:
process_counterfactual_csv("/Users/jonathanerskine/University of Bristol/gradient_supervision/counterfactual-gradient-alignment/Projects/SST-2/data/closs-output-sst_2-1000.csv")

✅ Saved splits to data/paired/ (train/dev/test)


In [ ]:
# Pipeline:
#  Define a neural net
#  Create a dataset to train on. Standard classifier example, or maybe XOR (or both)
#  Create a function script which enables either Cross-entropy or Direction loss function [ESSENTIAL]
#  Training model on the dataset on set of training points (very few -> 100% accuracy) [ESSENTIAL]
print("IMPORTING LIBRARIES")
## Standard libraries
import numpy as np
import pickle
import yaml

## Progress bar
from tqdm.auto import tqdm

#ML Libraries
import jax
import optax

import os

import torch.utils.data as data



# Custom Libraries
from counterfactual_alignment.custom_models   import SimpleClassifier, MLP, CNN,  GSPaper, GSPaperNew, GSPaper2, GSPaper3, BagOfWordsClassifier, BagOfWordsClassifierSimple, BagOfWordsClassifierSingle, SentimentModel
print("IMPORTS SUCCESSFUL")
from counterfactual_alignment.custom_datasets import customDataset
print("IMPORTS SUCCESSFUL")
from counterfactual_alignment.loss_functions import loss_functions
print("IMPORTS SUCCESSFUL")
from counterfactual_alignment.knowledge_functions import knowledge_functions
print("IMPORTS SUCCESSFUL")
from counterfactual_alignment.utilities import (visualise_classes, numpy_collate, imdb_collate,
                        reduce_dataset, compute_metrics, generate_results, generate_results_ensemble, create_train_state, boundary_filter, save_stats, train_one_epoch, combine_datasets)



IMPORTING LIBRARIES
IMPORTS SUCCESSFUL


25-Sep-06 17:52:51 fatf.utils.array.tools INFO     Using numpy's numpy.lib.recfunctions.structured_to_unstructured as fatf.utils.array.tools.structured_to_unstructured and fatf.utils.array.tools.structured_to_unstructured_row.


In [ ]:
from counterfactual_alignment.custom_datasets import customDataset

25-Sep-06 17:58:00 fatf.utils.array.tools INFO     Using numpy's numpy.lib.recfunctions.structured_to_unstructured as fatf.utils.array.tools.structured_to_unstructured and fatf.utils.array.tools.structured_to_unstructured_row.


In [ ]:



from scipy.stats import multivariate_normal as mvn
from sklearn import datasets as sk_datasets


25-Sep-06 17:59:15 fatf.utils.array.tools INFO     Using numpy's numpy.lib.recfunctions.structured_to_unstructured as fatf.utils.array.tools.structured_to_unstructured and fatf.utils.array.tools.structured_to_unstructured_row.


In [ ]:
from counterfactual_alignment.utilities import visualise_classes, expand_data, boundary_filter, jagged_lists_to_array, convert_to_list_of_lists

In [ ]:
from warnings import WarningMessage

import numpy as np
import pandas as pd
import pickle
import jax.numpy as jnp
import torch.utils.data as data
from counterfactual_alignment.utilities import visualise_classes, expand_data, boundary_filter, jagged_lists_to_array, convert_to_list_of_lists

In [1]:
from torch import normal
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence
import numpy as np
from scipy.stats import multivariate_normal as mvn

import jax
import jax.numpy as jnp
import optax
from flax.training import train_state
from flax import traverse_util
import flax
from flax import linen as nn

from tqdm.auto import tqdm

import random
import seaborn as sns
import sys
## Imports for plotting
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from matplotlib import colormaps

import time
import math
import subprocess
import glob
import os
import json
import csv
from pathlib import Path
from functools import partial

import counterfactual_alignment.custom_models as cm
from counterfactual_alignment.custom_models import custom_models

from matplotlib.ticker import FormatStrFormatter
from matplotlib.lines import Line2D


In [ ]:

from scipy.sparse import hstack, vstack

import tensorflow as tf
import pickle


25-Sep-06 18:01:49 fatf.utils.array.tools INFO     Using numpy's numpy.lib.recfunctions.structured_to_unstructured as fatf.utils.array.tools.structured_to_unstructured and fatf.utils.array.tools.structured_to_unstructured_row.


In [1]:

# FAT Forensics Counterfactual Explainer
from torch import normal
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence
import numpy as np
from scipy.stats import multivariate_normal as mvn

import jax
import jax.numpy as jnp
import optax
from flax.training import train_state
from flax import traverse_util
import flax
from flax import linen as nn

from tqdm.auto import tqdm

import random
import seaborn as sns
import sys
## Imports for plotting
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from matplotlib import colormaps

import time
import math
import subprocess
import glob
import os
import json
import csv
from pathlib import Path
from functools import partial

import counterfactual_alignment.custom_models as cm
from counterfactual_alignment.custom_models import custom_models

from matplotlib.ticker import FormatStrFormatter
from matplotlib.lines import Line2D


In [1]:

# FAT Forensics Counterfactual Explainer
import fatf.transparency.predictions.counterfactuals as fatf_cf

from scipy.sparse import hstack, vstack


25-Sep-06 18:09:12 fatf.utils.array.tools INFO     Using numpy's numpy.lib.recfunctions.structured_to_unstructured as fatf.utils.array.tools.structured_to_unstructured and fatf.utils.array.tools.structured_to_unstructured_row.


In [1]:

import tensorflow

: 